In [ ]:
#Requirements
import re
import numpy as np
import pandas as pd
import re
import string

#### Insert dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import files

In [ ]:
df = pd.read_csv('/my_path/Le_Μonde_RussoUkrainian_war_Clean.csv', encoding='utf-8')


In [ ]:
df.shape

In [ ]:
df.head(1)

In [ ]:
df.columns

In [ ]:
df['text_clean'][15]

### Τσκεκάρω κενές γραμμές και διπλές εγγραφές

In [ ]:
duplicates = df.duplicated(subset=['text_clean']).any()
duplicates

In [ ]:
df.shape

In [ ]:
df = df.drop_duplicates(subset=['text_clean'])

In [ ]:
df.shape

In [ ]:
Nan_sum = df['text_clean'].isnull().any()
Nan_sum

In [ ]:
df = df.dropna(subset=['text_clean'])

In [ ]:
df.shape

## Ξεκινάω προτασιοποιηση (sentecization)

In [ ]:
from tqdm import tqdm
import spacy

#  ελαφρύ μοντέλο  model
nlp = spacy.load("en_core_web_sm", disable=["ner", "tagger", "lemmatizer"])

#  parser για διαχωρισμό προτάσεων
nlp.enable_pipe("parser")

# tqdm για progress bar
tqdm.pandas()


In [ ]:
# μετατρέπω τα καθαρά κείμενα σε λίστα
texts = df['text_clean'].tolist()

# τρεχω με progress bar
docs = list(tqdm(nlp.pipe(texts, batch_size=20), total=len(texts)))


In [ ]:
# extract προτασιοποιμένα κειμενα

def extract_sentences(doc, filter_short=True):
    sents = [sent.text.strip() for sent in doc.sents]
    if filter_short:
        sents = [s for s in sents if len(s.split()) >= 3]
    return sents

In [ ]:
df['sents'] = [extract_sentences(doc) for doc in docs]


In [ ]:
df['sents'].head()

In [ ]:
# διαχωρισμός προτάσεων σε ξεχωριστές γραμμές με τη μέθοδο .explode
df = df.explode("sents", ignore_index=True)

In [ ]:
df.head(5)

In [ ]:
# μετονομάζω τη στήλη
df.rename(columns={"sents": "sents1"}, inplace=True)

In [ ]:
df.head(5)

In [ ]:
df.shape

In [ ]:
#check
df['sents1'][9384]

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 250)

In [ ]:
df['sents1'].head(10)

In [ ]:
# καθαρίζω κενά διαστήματα και αποθηκεύω σε στήλη 'sents_clean'
df['sents_clean'] = df['sents1'].str.strip()

In [ ]:
df['sents_clean'].head(10)

Αν κρίνω απαραίτητο κάνω επιπλέον καθαρισμό, ωστόσο τα μοντέλα transformers δουλεύουν καλά με πιο φυσικό κείμενο.

In [ ]:
# καθαρίζω πάλι

import re

def clean_text(text: str) -> str:

    # μετατρέπει το κείμενο σε string, για να αποφύγουμε errors με NaN
    text = str(text)

    # μετατρέπει όλα τα γράμματα σε πεζά
    text = text.lower()

    # αφαιρεί URLs / links
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # αντικαθιστά curly εισαγωγικά και αποστρόφους με απλά
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[’]', "'", text)

    # αντικαθιστά σημεία στίξης με κενό
    text = re.sub(r'[' + re.escape(string.punctuation) + r']', ' ', text)

    # αντικαθιστά παύλες με κενό
    text = re.sub(r'[–—−]', ' ', text)

        # αφαιρεί πολλαπλά κενά
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
#σωζω σε διαφορετική στήλη για να έχω προσβασή και στις δύο εκδοχές του κειμένου
df['sents_clean2'] = df['sents_clean'].apply(clean_text)

In [ ]:
pd.set_option('display.width', None)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', None)

In [ ]:
df['sents_clean2'].tail(50)

In [ ]:
#αποθηκεύω τον αριθμό λέξεων κάθε πρότασης σε δική του στήλη
df['num_wds'] = df['sents_clean'].apply(lambda x: len(x.split()))
df['num_wds'].mean()

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
#check
df['sents_clean'][5065]

### Τσεκάρω πάλι κενές γραμμές και διπλές εγγραφές




In [ ]:
nan_rows = df[df['sents_clean'].isna()]
nan_rows

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
#αφαιρώ Nan
df = df.dropna(subset=['sents_clean'])

In [ ]:
df.shape

In [ ]:
#τσεκάρω αν έχω διπλές εγγραφές
duplicates = df.duplicated(subset=['sents_clean']).any()
duplicates

In [ ]:
# ποσες διπλές εγγραφές
df.duplicated(subset=['sents_clean']).sum()

In [ ]:
# τυπώνω μέρικές διπλές για να ελέγξω
duplicates = df[df.duplicated(subset=['sents_clean'], keep=False)]  # keep=False τυπώνει όλες τις διπλές
duplicates

In [ ]:
df.shape

In [ ]:
# αφαιρώ διπλές εγγραφές
df = df.drop_duplicates(subset=['sents_clean'], keep='first')


In [ ]:
df.shape

In [ ]:
# κάνω reset το index και το αποθηκεύω ως Sent_id (=μοναδικό id για κάθε πρόταση) για να μην χάνω τις προτασεις μου
df = df.reset_index(drop=False).rename(columns={'index': 'Sent_id'})

In [ ]:
df.head(2)

In [ ]:
df.columns

In [ ]:
df.to_csv('/my_path/LeMonde_RussoUkrainian_Sents.csv', index=False)